<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/Flask%2Bngrok%E3%81%AE%E4%BE%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

セル1:インストールと authtoken 設定

In [ ]:
!pip install flask pyngrok -q

from pyngrok import ngrok, conf



トークンの指定

In [ ]:
# https://dashboard.ngrok.com/get-started/your-authtoken で取得
conf.get_default().auth_token = "2HhWtFPzcpgWIRTD67dJ7y59sYx_3nvbFfQgmH2JPJ7Aevg4t" #"ここにあなたのトークン"

ngrok は無料枠でも authtoken の登録が必須になっているので、ダッシュボードから取得して貼り付けてください。


セル2:Flask サーバを起動

In [ ]:
import threading
from flask import Flask, request
from pyngrok import ngrok

app = Flask(__name__)

@app.route("/")
def index():
    return """
    <h1>Colab + Flask + ngrok</h1>
    <p>動いています。</p>
    <form action="/echo" method="get">
      <input name="msg" placeholder="何か入力">
      <button>送信</button>
    </form>
    """

@app.route("/echo")
def echo():
    msg = request.args.get("msg", "")
    return f"受け取った文字列: {msg}"

# 既存トンネルがあれば閉じる(セル再実行対策)
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

# 443(HTTPS)で公開
public_url = ngrok.connect(5000, "http").public_url
print("公開URL:", public_url)

# Flask は別スレッドで起動(ノートブックをブロックしないため)
threading.Thread(
    target=lambda: app.run(port=5000, use_reloader=False),
    daemon=True
).start()

公開URL: https://4143-34-105-36-221.ngrok-free.app


印字された https://...ngrok-free.app をブラウザで開けば、443経由でアクセスできます。
いくつか補足です。ngrok.connect(5000, "http") の "http" は Colab 側(ローカル)のプロトコルで、外向きの公開URLは ngrok が自動でHTTPS(443)にしてくれます。無料枠は初回アクセス時に「警告ページ」を挟みますが、リクエストヘッダに ngrok-skip-browser-warning を付けるか、有料プランで消せます。スレッド起動にしているのは、app.run() をそのまま呼ぶとセルが固まって他の操作ができなくなるためです。